In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from itertools import combinations

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

def paired_ttest_report(name, a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]

    if len(a) < 2:
        return {
            "name": name,
            "n": len(a),
            "t": np.nan,
            "p": np.nan,
            "star": "NA"
        }

    t, p = stats.ttest_rel(a, b)
    return {
        "name": name,
        "n": len(a),
        "mean_a": float(np.mean(a)),
        "mean_b": float(np.mean(b)),
        "t": float(t),
        "p": float(p),
        "star": p_to_star(p)
    }

def welch_ttest_report(name, a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) < 2 or len(b) < 2:
        return {
            "name": name,
            "n1": len(a),
            "n2": len(b),
            "t": np.nan,
            "p": np.nan,
            "star": "NA"
        }

    t, p = stats.ttest_ind(a, b, equal_var=False)
    return {
        "name": name,
        "n1": len(a),
        "n2": len(b),
        "mean_1": float(np.mean(a)),
        "mean_2": float(np.mean(b)),
        "t": float(t),
        "p": float(p),
        "star": p_to_star(p)
    }

def onesample_ttest_report(name, a, popmean=1.0):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]

    if len(a) < 2:
        return {
            "name": name,
            "n": len(a),
            "t": np.nan,
            "p": np.nan,
            "star": "NA"
        }

    t, p = stats.ttest_1samp(a, popmean=popmean)
    return {
        "name": name,
        "n": len(a),
        "mean": float(np.mean(a)),
        "mu0": float(popmean),
        "t": float(t),
        "p": float(p),
        "star": p_to_star(p)
    }

jsonl_path = "swER_all.jsonl"
save_path = "fig_6def_smallworld.svg"



chosen_p = 0.2

net_order = ["FRP", "MOp", "Average"]

colors = {
    "FRP": "#d79036",
    "MOp": "#d8bf5a",
    "Average": "#4a9650",
    "Vanilla": "#8fbad3",
    "ER": "#b3b3b3",
}

records_by_net = {k: [] for k in net_order}

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)

        net = rec["net_type"]
        if net not in net_order:
            continue

        p = float(rec["p_thre"])
        if abs(p - 0.2) < 1e-9:
            records_by_net[net].append(rec)

def mean_sem(vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = vals.size
    if n == 0:
        return np.nan, np.nan
    if n == 1:
        return float(vals.mean()), 0.0
    sem = vals.std(ddof=1) / np.sqrt(n)
    return float(vals.mean()), float(sem)

C_mean, C_sem = [], []
C_er_mean, C_er_sem = [], []

L_mean, L_sem = [], []
L_er_mean, L_er_sem = [], []

S_mean, S_sem = [], []

for net in net_order:
    recs = records_by_net[net]

    C_obs = [r["sw_er"]["obs"]["C"] for r in recs]
    C_er  = [r["sw_er"]["rand"]["C_mean"] for r in recs]

    L_obs = [r["sw_er"]["obs"]["L"] for r in recs]
    L_er  = [r["sw_er"]["rand"]["L_mean"] for r in recs]

    S_obs = [r["sw_er"]["ratio"]["sigma_CL"] for r in recs]

    m, s = mean_sem(C_obs)
    C_mean.append(m); C_sem.append(s)

    m, s = mean_sem(C_er)
    C_er_mean.append(m); C_er_sem.append(s)

    m, s = mean_sem(L_obs)
    L_mean.append(m); L_sem.append(s)

    m, s = mean_sem(L_er)
    L_er_mean.append(m); L_er_sem.append(s)

    m, s = mean_sem(S_obs)
    S_mean.append(m); S_sem.append(s)

vanilla_C = 1.0
vanilla_C_sem = 1e-3
vanilla_C_er = 1.0
vanilla_C_er_sem = 1e-3

vanilla_L = 1.0
vanilla_L_sem = 1e-3
vanilla_L_er = 1.0
vanilla_L_er_sem = 1e-3

vanilla_sigma = 1.0
vanilla_sigma_sem = 1e-3

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))

group_labels = ["FRP", "MOp", "Average", "Vanilla"]
group_colors = [colors["FRP"], colors["MOp"], colors["Average"], colors["Vanilla"]]

x = np.arange(len(group_labels))
barw = 0.34

# -------- panel d: Clustering coefficient --------
C_obs_all = [C_mean[0], C_mean[1], C_mean[2], vanilla_C]
C_obs_sem_all = [C_sem[0], C_sem[1], C_sem[2], vanilla_C_sem]

C_er_all = [C_er_mean[0], C_er_mean[1], C_er_mean[2], vanilla_C_er]
C_er_sem_all = [C_er_sem[0], C_er_sem[1], C_er_sem[2], vanilla_C_er_sem]

axes[0].bar(
    x - barw/2, C_obs_all, barw,
    yerr=C_obs_sem_all, capsize=3,
    color=group_colors, edgecolor="none"
)
axes[0].bar(
    x + barw/2, C_er_all, barw,
    yerr=C_er_sem_all, capsize=3,
    color=colors["ER"], edgecolor="none"
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(
    ["FRP\nER$_{FRP}$", "MOp\nER$_{MOp}$", "Average\nER$_{Average}$", "Vanilla\nER$_{Vanilla}$"],
    rotation=-25
)
axes[0].set_ylabel("Clustering coefficient")

# -------- panel e: Average path length --------
L_obs_all = [L_mean[0], L_mean[1], L_mean[2], vanilla_L]
L_obs_sem_all = [L_sem[0], L_sem[1], L_sem[2], vanilla_L_sem]

L_er_all = [L_er_mean[0], L_er_mean[1], L_er_mean[2], vanilla_L_er]
L_er_sem_all = [L_er_sem[0], L_er_sem[1], L_er_sem[2], vanilla_L_er_sem]

axes[1].bar(
    x - barw/2, L_obs_all, barw,
    yerr=L_obs_sem_all, capsize=3,
    color=group_colors, edgecolor="none"
)
axes[1].bar(
    x + barw/2, L_er_all, barw,
    yerr=L_er_sem_all, capsize=3,
    color=colors["ER"], edgecolor="none"
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(
    ["FRP\nER$_{FRP}$", "MOp\nER$_{MOp}$", "Average\nER$_{Average}$", "Vanilla\nER$_{Vanilla}$"],
    rotation=-25
)
axes[1].set_ylabel("Average path length")

# -------- panel f: Small-world coefficient --------
sigma_labels = ["FRP", "MOp", "Average", "Vanilla"]
sigma_vals = [S_mean[0], S_mean[1], S_mean[2], vanilla_sigma]
sigma_errs = [S_sem[0], S_sem[1], S_sem[2], vanilla_sigma_sem]

axes[2].bar(
    np.arange(len(sigma_labels)), sigma_vals, 0.58,
    yerr=sigma_errs, capsize=3,
    color=[colors[k] for k in sigma_labels],
    edgecolor="none"
)
axes[2].set_xticks(np.arange(len(sigma_labels)))
axes[2].set_xticklabels(sigma_labels, rotation=-25)
axes[2].set_ylabel("Small-world coefficient")

for ax, lab in zip(axes, ["d", "e", "f"]):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(width=1.2)
    ax.text(-0.12, 1.02, lab, transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="bottom")

plt.tight_layout()
plt.savefig(save_path, format="svg", bbox_inches="tight", transparent=True)
plt.show()


# ======================
# ======================
print("\n===== Paired t-test: observed vs ER =====")

for net in net_order:
    recs = records_by_net[net]

    C_obs = [r["sw_er"]["obs"]["C"] for r in recs]
    C_er  = [r["sw_er"]["rand"]["C_mean"] for r in recs]

    L_obs = [r["sw_er"]["obs"]["L"] for r in recs]
    L_er  = [r["sw_er"]["rand"]["L_mean"] for r in recs]

    rep_C = paired_ttest_report(f"{net}: C vs ER", C_obs, C_er)
    rep_L = paired_ttest_report(f"{net}: L vs ER", L_obs, L_er)

    print(rep_C)
    print(rep_L)

# ======================
# ======================
print("\n===== Small-world coefficient tests =====")

sigma_by_net = {}
for net in net_order:
    recs = records_by_net[net]
    sigma_by_net[net] = np.array([r["sw_er"]["ratio"]["sigma_CL"] for r in recs], dtype=float)

for a, b in combinations(net_order, 2):
    rep = welch_ttest_report(f"sigma: {a} vs {b}", sigma_by_net[a], sigma_by_net[b])
    print(rep)

for net in net_order:
    rep = onesample_ttest_report(f"sigma: {net} vs Vanilla(=1)", sigma_by_net[net], popmean=1.0)
    print(rep)